In [ ]:
from vllm import LLM, SamplingParams
from pprint import pprint
from cs336_alignment.my_sft_toolfunc import *
from cs336_alignment.drgrpo_grader import r1_zero_reward_fn
import torch

In [ ]:
qwen_math = LLM(model="/home/nova/cs336/assignment5-alignment/models/Qwen2.5-Math-1.5B")

In [ ]:
sampling_params = SamplingParams(
    temperature=1.0,
    top_p=1.0, 
    max_tokens=1024, 
    stop=["</answer>"],
    include_stop_str_in_output = True
)

In [ ]:
qaiter = read_jsonl("/home/nova/cs336/assignment5-alignment/data/gsm8k/test.jsonl")
limited_qaiter = get_limited_iter(qaiter, 100)  # 把读文件限制设置在这里，防止一下读太多，后续操作均在可以设置这个限制的前提下进行
qas = get_qa_list(limited_qaiter)

In [ ]:
for qa in qas:
    a = qa["answer"]
    true_answer = a.split("#### ",1)[1] if "#### " in a else ""
    qa["ground_truth"] = true_answer

In [ ]:
prompts = []
with open("/home/nova/cs336/assignment5-alignment/cs336_alignment/prompts/r1_zero.prompt", encoding="utf-8") as f:
    template = f.read()

for qa in qas:
    q = qa["question"]
    prompts.append(template.format(question=q))

In [ ]:
request_outputs = qwen_math.generate(prompts,sampling_params=sampling_params)

In [ ]:
assert len(request_outputs) == len(qas) , "模型生成的回答数与投入的数据量不符？？"

In [ ]:
for i in range(len(qas)):
    r = request_outputs[i]
    qa = qas[i]
    assert r.prompt == template.format(question=qa["question"]), "输入的问题与输出中记录的不同？？"
    #print(generated_answer)
    
    generated_answer = r.outputs[0].text
    qas[i]["generated_answer"] = generated_answer

In [ ]:
output_path = "output.jsonl"

with open(output_path, "w", encoding="utf-8") as f:
    for record in qas:
        f.write(json.dumps(record, ensure_ascii=False))
        f.write("\n")

In [ ]:
i = 43
#print(qas[i])

for i in range(100):
    print(r1_zero_reward_fn(qas[i]["generated_answer"],qas[i]["ground_truth"]))
    print(template.format(question=qas[i]["question"]))
    print(qas[i]["generated_answer"])
    print("--------------------------------------------------")

In [ ]:
for qa in qas:
    reward = r1_zero_reward_fn(qa["generated_answer"],qa["answer"])
    qa.update(reward)

In [ ]:
qas[1]

In [ ]:
from typing import Callable
def evaluate_vllm(
    vllm_model: LLM,
    reward_fn: Callable[[str, str], dict[str, float]],
    prompts: list[str],
    eval_sampling_params: SamplingParams
) -> None:
    """
    对于已知的vllm大模型和采样参数，以及一组给定的输入，根据给定的奖励（打分）函数，评估模型回答的情况，并将结果序列化后存盘。
    """


In [ ]:
import transformers
print(transformers.__version__)